In [ ]:
# ============================================================
# Hotel Reviews Sentiment Analysis
# Notebook 1: Data Preprocessing
# Dataset: La Veranda Hotel - Booking.com Reviews
# ============================================================

# ============================================================
# SECTION 1: Install Libraries
# ============================================================

!pip install nltk
!pip install textblob
!pip install vaderSentiment
!pip install wordcloud
!pip install seaborn
!pip install gensim
!pip install joblib






In [ ]:
# ============================================================
# SECTION 2: Import Libraries
# ============================================================

import re
import string
import logging
import warnings

import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from textblob import TextBlob
from nltk.corpus import stopwords

warnings.filterwarnings('ignore')

In [ ]:
# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


In [ ]:
# ============================================================
# SECTION 3: Load Dataset
# ============================================================

df = pd.read_csv('Dataset/La_Veranda_Reviews-2023-01-16.csv')
df.head()

In [ ]:
# Check exact column names
print(df.columns.tolist())

In [ ]:
# ============================================================
# SECTION 4: Initial Exploration
# ============================================================

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nNull Values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
# ============================================================
# SECTION 5: Drop Unnecessary Columns
# Keep only what's needed for sentiment analysis
# ============================================================

# We keep: Positive Review, Negative Review, Score, 
#          Guest Country, Room Type, Number of Nights,
#          Visit Date, Group Type
# We drop: Guest Name (not analytically useful), 
#          Property Response (hotel's voice, not guest's)

df = df.drop(columns=['Guest Name', 'Property Response'], errors='ignore')
df.head()

In [ ]:
# ============================================================
# SECTION 6: Handle Missing Values
# ============================================================

# Some guests only leave a positive OR negative review — that's fine
# We fill empty reviews with empty string so cleaning doesn't break
df['PositiveReview'] = df['PositiveReview'].fillna('').astype(str)
df['NegativeReview'] = df['NegativeReview'].fillna('').astype(str)

mask = (df['PositiveReview'].str.strip() == '') & \
       (df['NegativeReview'].str.strip() == '')
df = df[~mask]

print("Remaining rows after dropping empty reviews:", len(df))
print("\nNull values after cleaning:\n", df.isnull().sum())

In [ ]:
# ============================================================
# SECTION 7: Text Cleaning Functions
# ============================================================

stop_words = set(stopwords.words('english'))

def remove_urls(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def remove_mentions_hashtags(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return re.sub(r'\@\w+|\#\w+', '', text)

def remove_punctuations(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_stopwords(text):
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    return ' '.join([w for w in text.split() if w not in stop_words])

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F700-\U0001F77F"
        u"\U0001F780-\U0001F7FF"
        u"\U0001F800-\U0001F8FF"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA00-\U0001FA6F"
        u"\U0001FA70-\U0001FAFF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def handle_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def lemmatize(text):
    lemmatizer = nltk.stem.WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

def stem(text):
    stemmer = nltk.stem.PorterStemmer()
    return ' '.join([stemmer.stem(word) for word in text.split()])

def clean_text(text):
    text = str(text).lower()
    text = remove_urls(text)
    text = remove_mentions_hashtags(text)
    text = remove_punctuations(text)
    text = remove_stopwords(text)
    text = remove_numbers(text)
    text = remove_emojis(text)
    text = handle_whitespace(text)
    text = lemmatize(text)
    text = stem(text)
    return text

In [ ]:
# ============================================================
# SECTION 8: Apply Cleaning to Both Review Columns Separately
# This is the key difference from the original AirBnb notebook —
# we treat positive and negative reviews as independent text streams
# ============================================================

df['cleaned_positive'] = df['PositiveReview'].apply(clean_text)
df['cleaned_negative'] = df['NegativeReview'].apply(clean_text)

df[['PositiveReview', 'cleaned_positive', 
    'NegativeReview', 'cleaned_negative']].head()

In [ ]:
# ============================================================
# SECTION 9: Score-Based Sentiment Labeling
# 
# Unlike the original which inferred sentiment purely from text,
# we use the Score column (1-10) as ground truth:
#   Score 1-4  → negative
#   Score 5-7  → neutral
#   Score 8-10 → positive
#
# We apply this separately for positive and negative review tracks
# ============================================================

def score_to_sentiment(score):
    if score <= 4:
        return 'negative'
    elif score <= 7:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(score_to_sentiment)
print(df['sentiment'].value_counts())

In [ ]:
# ============================================================
# SECTION 10: Also Compute TextBlob Polarity
# (for comparison/validation against Score-based labels)
# ============================================================

# Polarity on positive reviews
df['polarity_positive'] = df['cleaned_positive'].apply(
    lambda x: TextBlob(x).sentiment.polarity if x.strip() != '' else 0.0
)

# Polarity on negative reviews
df['polarity_negative'] = df['cleaned_negative'].apply(
    lambda x: TextBlob(x).sentiment.polarity if x.strip() != '' else 0.0
)

df[['Score', 'sentiment', 
    'polarity_positive', 'polarity_negative']].head(10)

In [ ]:
# ============================================================
# SECTION 11: Fix Date Column
# ============================================================

df = df[df['VisitDate'].str.contains('^\w', na=False)]
df['VisitDate'] = pd.to_datetime(df['VisitDate'], errors='coerce')
df = df.dropna(subset=['VisitDate'])

df['visit_month'] = df['VisitDate'].dt.month
df['visit_year'] = df['VisitDate'].dt.year

print("Date range:", df['VisitDate'].min(), "→", df['VisitDate'].max())

In [ ]:
# ============================================================
# SECTION 12: Remove Duplicates
# ============================================================

print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())

In [ ]:
# ============================================================
# SECTION 13: Final Dataset Overview
# ============================================================

print("Final shape:", df.shape)
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())
print("\nColumns:", df.columns.tolist())
df.head()

In [ ]:
# ============================================================
# SECTION 14: Save Cleaned Data
# Two files — one for positive review track, one for negative
# ============================================================

# Full cleaned dataset (used by ML/DL/LLM notebooks)
df.to_csv('Dataset/cleaned/hotel_reviews_cleaned.csv', index=False)
# Positive review track (for focused analysis)
pos_df = df[['cleaned_positive', 'polarity_positive', 
             'sentiment', 'Score', 'GuestCountry', 
             'RoomType', 'GroupType', 'visit_month', 'visit_year']]
pos_df.to_csv('Dataset/cleaned/positive_reviews_cleaned.csv', index=False)

# Negative review track
neg_df = df[['cleaned_negative', 'polarity_negative', 
             'sentiment', 'Score', 'GuestCountry', 
             'RoomType', 'GroupType', 'visit_month', 'visit_year']]
neg_df.to_csv('Dataset/cleaned/negative_reviews_cleaned.csv', index=False)

print("All files saved successfully!")